In [ ]:
%pip install -q datasets transformers tqdm xgboost


: 

In [ ]:
%pip install -q sentencepiece nltk

In [ ]:
from datasets import load_dataset


dataset = load_dataset('imdb')

# Quick stats
print(dataset)

# Convert to pandas for easier exploration
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

print('\nSample review:')
print(train_df.iloc[0].text[:500])
print('\nLabel:', train_df.iloc[0].label)

In [ ]:
import re
import nltk
nltk.download("punkt")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)         # remove HTML
    text = re.sub(r"[^a-z0-9\s]", ' ', text)    # remove punctuation
    text = re.sub(r"\s+", ' ', text).strip()
    return text

train_df['clean_text'] = train_df['text'].astype(str).map(clean_text)
test_df['clean_text'] = test_df['text'].astype(str).map(clean_text)

print("Sample cleaned text:", train_df['clean_text'].iloc[0][:300])


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

MAX_FEAT = 20000  # can increase or decrease for experiments (capacity ~ VC-dimension)
vectorizer = TfidfVectorizer(max_features=MAX_FEAT, ngram_range=(1,2), min_df=5)

X_train_tfidf = vectorizer.fit_transform(train_df['clean_text'])
X_test_tfidf = vectorizer.transform(test_df['clean_text'])

y_train = train_df['label'].values
y_test = test_df['label'].values

print("TF-IDF shape:", X_train_tfidf.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_tfidf, y_train)

y_pred = logreg.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(max_iter=5000)
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=None)  # unlimited depth = very high VC-dimension
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

def vc_dimension_experiment(max_features_list=[500, 1000, 5000, 10000 , 15000, 20000, 30000]):
    results = []
    for mf in max_features_list:
        vec = TfidfVectorizer(max_features=mf, ngram_range=(1,2), min_df=5)
        Xtr = vec.fit_transform(train_df["clean_text"])
        Xte = vec.transform(test_df["clean_text"])
        lr = LogisticRegression(max_iter=500)
        lr.fit(Xtr, y_train)
        acc_train = accuracy_score(y_train, lr.predict(Xtr))
        acc_test = accuracy_score(y_test, lr.predict(Xte))
        results.append((mf, acc_train, acc_test))
        print(f"max_features={mf}, Train Acc={acc_train:.3f}, Test Acc={acc_test:.3f}")
    return results

results = vc_dimension_experiment()

# Plot
xs = [r[0] for r in results]
train_acc = [r[1] for r in results]
test_acc = [r[2] for r in results]

plt.plot(xs, train_acc, marker="o", label="Train Acc")
plt.plot(xs, test_acc, marker="s", label="Test Acc")
plt.xlabel("TF-IDF max_features (proxy for VC-dimension)")
plt.ylabel("Accuracy")
plt.title("Effect of VC-Dimension on Overfitting")
plt.legend()
plt.show()
